<a href="https://colab.research.google.com/github/nanpolend/machine-learning/blob/master/notebookaa7aGPT5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

arc_prize_2025_path = kagglehub.competition_download('arc-prize-2025')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import argparse, sys, json, random, time
import numpy as np
from pathlib import Path

# ====== 隨機種子設定 ======
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ====== 判斷 Notebook / CLI 環境 ======
def get_args():
    if "ipykernel" in sys.modules:  # Notebook 或 Colab
        print("[INFO] 偵測到 Notebook 環境，自動載入預設參數")
        return argparse.Namespace(
            data_root="/content/arc-data",   # <-- 修改為你的資料路徑
            split="public",
            out="submission.json",
            beam=10,
            depth=4,
            cands=50,
            timeout=8.0
        )
    else:  # CLI 模式
        parser = argparse.ArgumentParser()
        parser.add_argument("--data-root", required=True, help="ARC 資料集路徑")
        parser.add_argument("--split", default="public", help="public / private")
        parser.add_argument("--out", default="submission.json", help="輸出檔名")
        parser.add_argument("--beam", type=int, default=10)
        parser.add_argument("--depth", type=int, default=3)
        parser.add_argument("--cands", type=int, default=50)
        parser.add_argument("--timeout", type=float, default=8.0)
        return parser.parse_args()

args = get_args()

# ====== 微型 DSL 原語 ======
def rotate90(grid): return np.rot90(grid)
def flip_h(grid): return np.fliplr(grid)
def flip_v(grid): return np.flipud(grid)
def identity(grid): return grid

PRIMITIVES = [identity, rotate90, flip_h, flip_v]

# ====== Beam Search 搜尋器 ======
class BeamSearchSolver:
    def __init__(self, beam_width=10, max_depth=3, timeout=8.0):
        self.beam_width = beam_width
        self.max_depth = max_depth
        self.timeout = timeout

    def solve(self, train_pairs, test_input):
        start_time = time.time()
        beam = [(test_input, [])]

        for depth in range(self.max_depth):
            new_beam = []
            for grid, prog in beam:
                for prim in PRIMITIVES:
                    out_grid = prim(grid)
                    if all(np.array_equal(prim(inp), outp) for inp, outp in train_pairs):
                        return out_grid
                    new_beam.append((out_grid, prog + [prim]))
            # 排序 + 保留前 beam_width 個候選
            beam = sorted(new_beam, key=lambda x: random.random())[:self.beam_width]
            if time.time() - start_time > self.timeout:
                break
        return beam[0][0]

# ====== 主程式 ======
def main():
    solver = BeamSearchSolver(
        beam_width=args.beam,
        max_depth=args.depth,
        timeout=args.timeout
    )

    dataset_dir = Path(args.data_root)
    public_tasks = list((dataset_dir / args.split).glob("*.json"))

    submission = {}
    for task_file in public_tasks:
        task_id = task_file.stem
        with open(task_file) as f:
            task = json.load(f)
        train_pairs = [(np.array(p["input"]), np.array(p["output"])) for p in task["train"]]
        test_input = np.array(task["test"][0]["input"])
        output = solver.solve(train_pairs, test_input)
        submission[task_id] = output.tolist()

    with open(args.out, "w") as f:
        json.dump(submission, f)
    print(f"[INFO] 已輸出 {args.out}")

if __name__ == "__main__":
    main()
